## Ed-Tech Quiz

In [5]:
import json
from dotenv import load_dotenv 
from langchain.llms import OpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains import SequentialChain
import streamlit as st
import traceback 
import pandas as pd 
from langchain.callbacks import get_openai_callback 
from utils import parse_file, get_table_data, RESPONSE_JSON

load_dotenv()

# This is an LLMChain to create 10-20 multiple choice questions from a given piece of text.
llm = OpenAI(model_name = "gpt-3.5-turbo", temperature=0, max_toekns=-1)

template = """
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to\
create a quiz of {number} multiple choice questions for grade {grade} students in {tone} tone.
Make sure that questions are not repeated and check all the questions to be conforming to the text as well.
Make sure to format your response like the RESPONSE_JSON below and use it as a guide.\
Ensure to make the {number} MCQSs.
### RESPONSE_JSON
{response_json}
"""

quiz_generation_prompt = PromptTemplate(
    input_variables = ["text", "number", "grade", "tone", "response_json"],
    template = template,
)
quiz_chain = LLMChain(
    llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True
)

# This is an LLMChain to evalaute the multiple choice questions created by the above chain
llm = OpenAI(model_name = "gpt-3.5-turbo", temperature=0)
template = """You are an expert english grammarian and writer. Given a multiple choice quiz for {grade} grade students.\
You need to evaluate complexity of the questions and give a complex analysis of the quiz if the students
will be able to understand the questions and answer them. Only use at max 50 words for compelxity analysis.
If quiz is not at par with the cognitive and analytical abilities of the students,\
update the quiz questions which need to be changed and change the tone such that is perfectly fits the students abilities.
Quiz MCQS:
{quiz}
Critique from an expert english writer of the above quiz:""""

quiz_evaluation_prompt = PromptTemplate(
    input_variables = ["grade", "quiz"], template = template
)
review_chain = LLMChain(
    llm=llm, prompt_quiz_evaluation_prompt, output_key="review", verbose=True
)

# This is the overall chain where we run these two chains in sequence.
generate_evaluate_chain = SequentialChain(
    chains = [quiz_chain, review_chain],
    input_variables = ["text", "number", "grade", "tone", "response_json"],
    # Here we return multiple variables
    output_variables = ["quiz", "review"],
    verbose = True,
)

st.title("Quiz Generation for Educational Content")

# Create a form using st.form 
with st.form("user_inputs"):
    # File upload 
    uploaded_file = st.file_uploader("Upload a pdf or text file")

    # Input fields
    mcq_count = st.number_input("No of MCQs", min_value=3, max_value=20)
    grade = st.number_input("Inser Grade", min_value=1, max_value=10)
    tone = st.text_input("Insert Quiz tone", max_chars=100, placeholder="simple")

    button = st.form_submit("Create quiz")

# Check if the button is clicked and all fields have inputs 
if button and uploaded_file is not None and mcq_coun and grade and tone:
    with st.spinner("Loading..."):
        try:
            text = parse_file(upload_file)

            # Count tokens and cost of api call
            with get_openai_callback() as cb:
                response = generate_evaluate_chain(
                    {
                        "text": text,
                        "number": mcq_count,
                        "grade": grade,
                        "tone": tone,
                        "response_json": json.dumps(RESPONSE_JSON),
                    }
                )
        except Exception as e:
            traceback.print_exception(type(e), e, e.__traceback__)
            st.error("Error")
        else:
            print(f"Total Tokens: {cb.total_tokens}")
            print(f"Prompt Tokens: {cb.prompt_tokens}")
            print(f"Completion Tokens: {cb.completion_tokens}")
            print(f"Total Cost (USD): ${cb.total_cost}")

            if isinstance(response, dict):
                # Exract quiz data from the response 
                quiz = response.get("quiz", None)
                if quiz is not None:
                    table_data = get_table_data(quiz)
                    if table_data is not None:
                        df = pd.DataFrame(table_data)
                        df.index = df.index + 1
                        st.table(df)
                        # Display the review in a text box 
                        st.text_area(label="Review", value_response["review"])
                    else:
                        st.error("Error in table data")
                else:
                    st.write(response)

SyntaxError: unterminated string literal (detected at line 46) (465119755.py, line 46)

## YouTube Content Ideas

In [ ]:
import asyncio
from langchain.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain 
from langchain.text_splitter import RecursiveCharacterTextSplitter 
from langchain.document_loaders import YoutubeLoader 

imort pandas as pd

class Summarizer:
    def __init__(self, url, llm):
        self.url = url
        self.llm = llm
        self.doc_chunks = []
        self.metadata = []

    def _load_data(self):
        loader = YoutubeLoader.from_youtube_url(self.url, add_video_info=True)

        return loader.load()

    def create_chunks(self):
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=3500,
            chunk_overlap=20,
            length_function=len,
        )
        text = self._load_data()
        self.doc_chunks = text_splitter.create_documents(
            [doc.page_content for doc in text]
        )
        self.metadat = text[0].metadat

        return 

    async def _chain_run(self, chain, docs):
        return await chain.arun(docs)

    async def summarize(self):
        summarizer_chain = load_summarize_chain(llm=self.llm, chain_type="map_reduce")
        tasks = [self._chain_run(summarize_chain, self.doc_chunks)]
        summary = await asyncio.gather(*tasks)

        return {"summary": summary[0], "metadata": self.metadata}

In [ ]:
import concurrent.features
import asyncio
import time 

def process_summary(url):
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5)
    summarizer = Summarizer(url, llm)
    summarizer.create_chunks()

    return asyncio.run(summarizer.summarize())

def pool_executor(url):
    result = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Submit each URL to complete 
        futures = [execute.sunmit(process_summary, url) for url in urls]

        # Wait for all futures to complete
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                print(f"Error processing URL: {e}")
    return results

In [ ]:
high_engagement_vids_urls = [
    "https://www.youtube.com/watch?v=_8bMMqy37y8&ab_channel=ThisWeekinStartups",
    "https://www.youtube.com/watch?v=1SWEF-lyW28&ab_channel=ThisWeekinStartups",
    "https://www.youtube.com/watch?v=oc5tHbEK0IQ&ab_channel=ThisWeekinStartups",
    "https://www.youtube.com/watch?v=jrd4snFDSVA&ab_channel=ThisWeekinStartups",
]

low_engagement_vids_urls = [
    "https://www.youtube.com/watch?v=UeIV4KcSUlk",
    "https://www.youtube.com/watch?v=hNcLMN_bZCM",
    "https://www.youtube.com/watch?v=ANd4jPLnMAU",
    "https://www.youtube.com/watch?v=J8YnxrGEzT4",
]

high_engagement_summaries = pool_executor(high_engagement_vids_urls)
low_engagement_summaries = pool_executor(low_engagement_vids_urls)

In [ ]:
high_eng_df = pd.DataFrame.from_records(high_engagement_summaries)
high_eng_df

In [ ]:
low_eng_df = pd.DataFrame.from_records(low_engagement_summaries)
low_eng_df

In [6]:
def format_summaries(data):
    formatted_strings = []
    for i, obj in enumerate(data, start=1):
        summary = obj["summary"]
        views = obj["metadata"]["view_count"]
        title = obj["metadata"]["title"].split("|").[0].strip()

        formatted_string = (
        f"Video {i}\nTitle: {title}\nView Count: {views}\nSummary: {summary}\n"
        )
        formatted_strings.append(formatted_string)
    result = "\n".join(formmatted_strings)
    return result 

high_eng_prompt = format_summaries(high_engagement_summaries)
low_eng_prompt = format_summaries(low_engagement_summaries)

_IncompleteInputError: incomplete input (2660473814.py, line 5)

In [ ]:
print(high_eng_prompt)

In [ ]:
print(low_eng_prompt)

In [ ]:
from langhain.prompt import PromptTemplate 
from langchain.chains import LLMChain 
from langchain.chat_models import ChatOpenAI

prompt_template = """ You are helpful AI assistant that helps to increase the engagement of youtube videos by analyzing the scripts of old videos.\
Looking at the given videos below in High_engagement_Videos and Low_engagement_videos sections,\
come up with new ideas for next videos.\

High_Engagement_Videos:
{high_engagement_videos}

Low_Engagement_Videos:
{low_engagement_videos}

Given the above High_Engagement_Videos and Low_Engagement_Videos, generate new ideas and themes.
New ideas should be realed to the High_Engagement_Videos by keeping the titles and summaries of the episodes in context and must be based on\
the common patterns between the High_Engagement_Videos and the guests in those episodes.\
The new videos should not have any content from Low_Engagement_Videos.\
Make sure to not include any speaker name in your suggested video topics or themes.
Make sure to retunr at least 10 new ideas. Your response must be a csv file which contains the following columns:\
Topic, Theme, Summary. Summary should contain points to talk on the show and must be 100 words at max. Use | as a speaker\
and do not append any extra line in your csv response. Each row must have proper data and columns in each row must be three.
If you don't know the answer, just say "Hmm, I'm not sure."\
Don't try to make up an answer.
"""
ll m = ChatOpenAI(model = "gpt-3.5-turbo", temperature = 0.5)
PROMPT = PromptTemplate(
    template = prompt_template,
    input_variables = ["high_engagement_videos", "low_engagement_videos"],
)
ideas_chain = LLMChain(llm=llm, prompt=PROMPT, verbose=True)

In [ ]:
response = ideas_chain.run(
    {
        "high_engagement_videos": high_eng_prompt,
        "low_engagement_videos": low_eng_prompt,
    }
)

In [ ]:
response

In [ ]:
import pandas as pd
from io import StringIO

csv_file = StringID(response)

# Read the CSV data and create a DataFrame
df = pd.read_csv(csv_file, sep="|")

In [ ]:
df

## Time Table for Campus Classes

In [ ]:
import os 
import openai 
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
import csv 

def read_data_from_csv(filename):
    data = []

    with open(filename, 'r') as csvfile:
        return list(csv.DictReader(csvfile))

In [ ]:
subjects_data = read_data_from_csv("subjects.csv")
students_data = read_data_from_csv("students.csv")

In [ ]:
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
chat = ChatOpenAI(model="gpt-4", temperature=0)

system_message_prompt = SystemMessagePromptTemplate.from_template(
    "You are an expert university class scheduler with proficiency in interpreting JSON."
)
human_message_prompt = HumanMessagePromptTempalte.from_template(
"""
The following subjects data contains subject names, the total number of classes to be attended by a student, and the availabe class slot for each subject.
{subjects_data}

Here is the data for a specific student, including his registered subjects and considerations. Your task is to schedule classes for these subjects.
{student}

The number of classes per week for each subject must be the same as mentioned in the data. You must make sure that classes should not overlap.\
If there is a conflict between the student's considerations and the scheduling, ignore the consideration. Include a brief comment at the end regarding\
the extent to which the considerations have been met, specifying the suject names.

The timetable should be prepared in the following format:
StudentName:
Subject1:
Classes list 
Subject2:
Classes list 
Subject3:
Classes list
...

Remember that classes must not overlap, you can ignore considerations when needed.
"""
)

chat_prompt = ChatPromptTempalte.from_messages([system_message_prompt, human_message_prompt])
chain = LLMChain(llm=chat, prompt=chat_prompt)

In [ ]:
for student in students_data:
    response = chain.run(subjects_data = subjects_data, student_data, student=student)
    print(response, '\n')

## Speech Synthesis

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI
from elevenlabs import generate, Voice, voices 
import streamlit as st 
import os 

def generate_intro(topic, theme, summary, show):
    prompt_template = """" As an expert writer, your task is to create an introduction for a captivating podcast that will leave \
    the listener spellbound and eager to explore more. Your audience is diverse, and your words should response with \
    their curiosity, emotions and interests. Following are the steps to guide you in creating the perfect introduction:

    TOPIC: {topic}
    THEME: {theme}
    SUMMARY: {summary}

    Looking at above TOPIC, THEME and SUMMARY, your mission is to weave a magical introduction in the style of {show} that effortlessly draws \ 
    the listeners into the heart of the narrative. The introduction must be engaging, \
    thought-provoking, and leave an indelible mark on the minds of those who tune in.

    Consider the following guidelines while composing your introduction:
    Begin with a captivating hook: Capture the audience's attention right from the start with a mesmerizing opening line \
    that sparks curiosity and creates an irresitible urge to explore further.

    Set the tone: Infuse the introduction with an appropriate tone that complements the podcat's theme, \
    be it mysterious, uplifting, nostalgic, or thrilling.

    Paint vivid mental images: Use your mastery of langauge to create beautiful visualizations and \
    paint a world that lures listeners into the realms of the podcast's subject matter.

    Appeal to emotions: Connect with the audience's emotions by incorporating elements that evoke \
    empathy, excitement, wonder, or even nostalgia.

    Unveil the essence: While keeping the mastery alive, provide a glimpse of what the podcast entails, \
    leaving the listeners yearning for more insights and revelations.

    Embrace diversity: Ensure your writing is inclusive, relatable, and appeals to a broad range \
    of audiences, transcending age, culture, and background.

    Be concise yet impactful: Craft a concise introduction that leaves a lasting impact, \
    making every word count towards creating an enchanting experience.
    Embrace your role as an expert writer, and let your creativivty and linguistic prowess shine through \
    in this introductory piece. Never break your character. The Introduction must be \
    300 words at max. The podcast introduction must be in style of {show}
    """
    llm = ChatOpenAI(model = "gpt-3.5-turbo", temperature=0.1)
    PROMPT = PromptTemplate(
        template=prompt_template,
        input_variables=["topic", "theme", "summary", "show"],
    )
    chain = LLMChain(llm=llm, prompt=PROMPT, verbose=True, output_key="introduction")
    response = chain({"topic": topic, "theme": theme, "summary": summary, "show": show})
    return response.get("introduction")

def generate_audio(intro, voice):
    return generate(text=intro, voice=voice, model="eleven_monolingual_v1")

def get_image_path(speaker):
    image_path = f"./images/{speaker.lower().replace(' ', '-')}-profile.jpg"
    if os.path.exists(image_path):
        return image_path
    else:
        return None

@st.cache_data(show_spinner=False)
def get_voices():
    speakers = [voice.name for voice in voices()]
    speakers.insert(0, "")
    return speakers

def get_voice_by_name(name):
    return [voice for voice in voices() if voice.name == name][0]

In [ ]:
from utils import (
    generate_intro,
    generate_audio,
    get_image_path,
    get_voices,
    get_voice_by_name,
)
import streamlit as st 
from dotenv import load_dotenv

load_dotenv()

def main():
    st.markdown(
        "<h1 style='text-align: center;'><b>Speech Generator</b></h1>",
        unsafe_allow_html=True,
    )

    # Create a form with three input fields
    with st.form("intro_form"):
        topic = st.text_input("Topic (Max 300 characters)", max_chars=300)
        theme = st.text_area("Theme")
        summary = st.text_area("Summary")
        show = st.selectbox(
            "Select podcat style", 
            ["The Joe Rogan Experience", "The Jordan B. Peterson Podcast"],
        )
        generate_intro_button = st.form_submit_button("Generate Intro")

        if generate_intro_button and all([topic, theme, summary, show]):
            with st.spinner("Generating Intro..."):
                # Call the function to generate the intro and get the output
                podcast_intro = generate_intro(topic, themes, summary, show)
                st.session_state.podcast_intro = podcast_intro
        elif generate_intro_button and not all([topic, theme, summary, show]):
            st.warning("Fill all the above field.")
    if "podcast_intro" in st.session_state:
        podcast_intro = st.session_state.podcast_intro
        st.header("Podcast Introduction")
        st.write(podcast_intro)

    if "selected_speaker" not in st.session_state:
        st.session_state.selected_speaker=""

    if "podcast_intro" in st.session_state:
        st.header("Speech Synthesis")
        speakers = get_voices()
        speaker = st.selectbox("Select a Speaker", speakers, index=0)

        # Create a button "Create" to generate audio
        if speaker and speaker != "":
            st.session_state.selected_speaker = speaker
            if speaker_image_path = get_image_path(speaker)
                st.image(speaker_image_path, caption=speaker, use_column_width=True)
            with st.spinner("Getting Voice from ElevenLabs..."):
                speaker_voice = get_voice_by_name(speaker)
            with st.spinner("Generating audio..."):
                audio_bytes = generate_audio(podcast_intro, speaker_voice)
            if audio_bytes:
                st.success("Audio Generated")
                st.audio(audio_bytes, format="audio/wav")
            else:
                st.error("Speaker name must be selected")

if __name__==""__main__"":
    main()

## YouTube Content Ideas From Trending Post

In [ ]:
import lxml.html 
import requests 

def refactor_extracted_comments(comments: list) -> str:
    """
    Refactors and cleans the extracted comments

    :param comments: comments being refactored
    :return: refactored comments
    """
    comments = "".join(comments)
    lines = comments.split("\n")
    cleaned_lines = [line.strip() for line in lines]

    refactored_lines = [line for line in cleaned_lines if line]
    refactored_text = "\n".join(refactored_lines).replace("\n\n", "\n").replace("reply", "\nreply:")

    return refactored_text

def extract_hackernews_page_content(url: str) -> dict:
    """
    Extracts the page content from the specified url

    :param url: url for which content is exracted
    :return: extracted page content
    """
    response = requests.get(url=url)

    page = lxml.html.fromstring(response.text)
    title_text = page.xpath("//span[@class='titleline']//a/text()")
    description_text = page.xpath("//div[@class='toptext']/text()")
    comments = page.xpath("//div[@class='comment']//descendent-or-self::*/text()")

    refactored_comments = refactor_extracted_comments(comments)

    return {
        "title": "".join(title_text),
        "description": "".join(description_text),
        "comments": refactored_comments
    }

In [ ]:
import os 

from dotenv import load_dotenv
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)
load_dotenv()

chat = ChatOpenAI(openai_api_key = os.getennv("OPENAI_API_KEY"))

def generate_youtube_ideas_from_content(page_content: dict) -> str:
    """
    Generate youtube ideas for the given content
    :param page_content: extracted title, description and comments/replies
    :return: generated ideas
    """
    human_message = """
    I've the following topic
    {title}
    with the description
    {description}
    and the following comments on that
    {comments}

    Generate 5 different viral YouTube content ideas realted to this. For each idea, please provide
    title, description and youtube content script (with timestamps)

    Output the result in the following format using markdown:

    Idea 1:
    Title:
    Description:
    YouTube content script(with timestamps):
    0:00 some script content
    0:30 other script content

    Idea 2:
    Title:
    Description:
    YouTube content script (with timestamps):
    0:00 some script content 
    0:30 other script content 
    ...
    """

    system_message_prompt = SystemMessagePromptTemplate.from_template("You are an expert youtube contnt creator.")
    human_message_prompt = HumanMessagePromptTemplate.from_template(human_message)

    chat_prompt = ChatPromptTemplate.from_messages(
        [
            system_message_prompt,
            human_message_prompt
        ]
    )
    chat_model = ChatOpenAI(temperature=0.2, model="gpt-3.5-turbo-16k")

    llm_chain = LLMChain(prompt=chat_prompt, llm=chat_model)
    response = llm_chain.run(
        title=page_content["title"],
        description=page_content["description"],
        comments=page_content["comments"]
    )
    response = response.replace("\n", "\n\n")

    return response

In [ ]:
import streamlit as st 

from extract_post_content import extract_hackernews_page_content 
from generate_youtube_content_ideas import generate_youtube_ideas_from_content 

def main():
    """
    Function to execute the script
    """
    st.title("YouTube Viral Video Ideas Generator")

    url = st.text_input("Enter the URL:")

    if url:
        with st.spinner("Processing..."):
            page_content = extract_hackernew_page_content(url)
            response = generate_youtube_ideas_from_content(page_content)

            st.markdown(response)
    else:
        st.warning("Please enter a URL first.")

if __name__ == "__main__":
    main()

## BedTime Story Teller

In [ ]:
import streamlit as st
from dotenc import load_dotenv

from utils import generate_child_fields, get_voice_by_name, generate_audio, generate_bedtime_story

load_dotenv()

def main():
    st.title("Bed Story Generator")

    st.write(
        "Please enter the details about the children to create a bedtime story."
    )
    child_1_details = generate_child_fields(1)
    child_2_details = generate_child_fields(2)

    st.write("Further details")
    children_relation = st.text_input("Relation between children*")

    children_data = {
        "children": [child_1_details, child_2_details],
        "relation between children": children_relation,
    }

    required_fields_filled = all(child_data is not None for child_data in children_data["children"])

    if st.button("Generate Bedtime Story") and required_fields_filled and children_relation:
        st.subheader("Bedtime Story")

        with st.spinner("Processing..."):
            st.session_state.bedtime_story = generate_bedtime_story(children_data)

    if "bedtime_story" in st.session_state:
        st.write(st.session_state.bedtime_story)
        speaker_type = st.selectbox("Select the Speaker Voice Type", ["Male", "Female"])

        if st.button("Generate Audio"):
            speaker = "Thomas" if speaker_type == "Male" else "Dorothy"

            with st.spinner("Getting Voice from ElevenLabs..."):
                speaker_voice = get_voice_by_name(speaker)

            with st.spinner("Generating audio..."):
                audio_bytes = generate_audio(st.session_state.bedtime_story, speaker_voice)

            if audio_bytes:
                st.success("Audio Generated")
                st.audio(audio_bytes, format="audio/wav")
    else:
        st.warning("Please fill in all the required fields before genertaing the bedtime story.")

if __name__ == "__main__":
    main()

In [7]:
import os 

import streamlit as st
from elevenlabs import voices, generate
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePrompt, 
    HumanMessagePromptTemplate,
)

def generate_bedtime_story(children_data):
    os.enviorn["OPEN_API_KEY"] = os.getenv("OPENAI_API_KEY")
    chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

    system_message_prompt = SystemMessagePromptTemplate.from_template(
        """As a gifted author of bedtime stories for children, your tales consistently embody these captivating elements:
1. Intriguing climaxes and surprising anti-climaxes
2. Thrilling adventures
3. A series of engaging and challenging tasks for children.
4. Detailed descriptions of each task and the steps taken to accomplish them.
5. Stimulate problem-solving skills in young readers.
6. Introduce a multitude of imgignative creatures with varying powers.
7. Accompany the story with colorful illustrations to maintain interest and aid understanding.
8. Incorporate recurring sequences or phrases that soothe and reassure young listeners.
9. Engage young readers with elaborative descriptions that captivate their imagination
10. Infuse positive affirmations throughout the story to encourage confidence and kindness.
11. Provide a heartwarming happy ending with valueable life lessons to inspire young minds.
12. Include a variety of problems with climaxes and anti-climaxes for added excitement
13. Employ a pleasant tone that soothes the heart and minds of young readers.
14. Demonstrate verbosity and innovation in storytelling to keep the narrative fresh and captivating.
"""
    )

    human_message_prompt = HumanMessagePromptTemolate.from_template(
        """ Below are the profiles of two children:
{children_data}

Your enchanting taks is to craft a mesmerizing detailed and descriptive bedtime story that contains these characters.
You must include each child's characteristics given in the above data, into the story.

Embrace the magic of storytelling and create a unique unforgettable innovative tale that imparts wisdom, ignites imagination, encourages action\
with task and problem solving, and leaves a lasting impression on their young hearts.
You must describe each task or adventure they accomplish in very detail. The story must be descriptive.
"""
    )
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])
    chain = LLMChain(llm=chat, prompt=chat_prompt, verbose=True)

    response = chain.run(children_data=children_data)
    return response 

def generate_child_fields(child_number):
    st.subheader(f"Child {child_number} Details")
    name = st.text_input(f"C{child_number}. Name*")
    age = st.number_input(f"C{child_number}. Age*", min_value=1, max_value=100)
    interests = st.text_area(f"C{child_number}. Interests (separated by commas)*")
    superpowers = st.text_area(f"C{child_number}. Superpowers loved (separated by commas)*")
    challenges_fears = st.text_input(f"C{child_number}. Challenges/Fears*")

    dream_destination = st.text_input(f"C{child_number}. Dream Destination")
    hobbies = st.text_area(f"C{child_number}. Hobbies/Activities (separated by commas)")
    best_person_name = st.text_input(f"C{child_number}. BBests person name")
    best_person_relation = st.text_input(f"C{child_number}. Best person relation")
    favorite_book_relation = st.text_input(f"C{child_number}. Facorite Book/Movie")
    favorite_food = st.text_input(f"C{child_number}. Favorite Food")

    required_fields = [name, gender, age, interests, superpowers, challenges_fears]
    if not all(required_fields):
        return None

    return {
        "Name": name,
        "Gender": gender,
        "Age": age,
        "Interests": interests.spit(","),
        "Superpowers loved": superpowers.split(","),
        "Dream Destination": dream_destination,
        "Challenges/Fears": challenges_fears,
        "Hobbies/activities": hobbies.split(","),
        "Best person name": best_person_name,
        "Relation with best person": best_person_relation,
        "Favorite Food": favorite_food,
        "Favorite Book/Movie": favorite_book_movie,
    }

@st.cache_data(show_spinner=False)
def get_voice_by_name(name):
    return next((voice for voice in voices() if voice.name == name), None)

def generate_audio(intro, voice):
    return generate(text=intro, voice=voice, model="eleven_monolingual_v1")

SyntaxError: unterminated string literal (detected at line 77) (3083029487.py, line 77)

## SQL ToolKit

In [ ]:
from langchain.agents import create_sql_agent 
from langchain.agents.agent_toolkits import SQLDatabaseToolkil
from langchain.sql_database import SQLDatabase
from langchain.llms.openai import OpenAI

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain_visualizer import visualize 
from langchain.callbacks import get_openai_callback 
import langchain 
import os 

langchain.debug = True 

db_path = os.path.abspath("./orders.db")

db = SQLDatabase.from_uri(
    f"sqlite:////{db_path}",
    include_tables=["return_policy", "category", "product", "order"],
    sample_rows_in_table_info=,
)
print(db.table_info)

In [ ]:
llm = OpenAI(temperature=0)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    max_iterations=30,
    return_intermediate_steps=True,
)

def agent_executor(query):
    with get_openai_callback() as cb:
        response = agent.run(query)
        print(f"Total Tokens: {cb.total_tokens}")
        print(f"Prompt Tokens: {cb.prompt_tokens}")
        print(f"Completion Tokens: {cb.completion_tokens}")
        print(f"Total Cost (USD): ${cb.total_cost}")
        return response

In [ ]:
query = "List all the relation names"
agent_executor(query)

In [ ]:
agent_executor("describe the category table")

In [ ]:
agent_executor("How many orders are in order table?")

In [ ]:
agent_executor("Describe order whose id is 16. What is the product and its category?")

In [ ]:
agent_executor(
    "My order id is 16. What is the product's name and return policy related to this order?"
)

In [ ]:
agent_executor("My order id is 12. Can I return the product today?")